In [1]:
import pandas as pd
import numpy as np

This notebook prepares the datasets for input to ShatterSeek (creating two files for the SV & CNV data)

In [ ]:
# Loading & filtering data
cnv_file = pd.read_hdf('../../SV_sample_data/data/analysis_public_os_data_merged_abs_segfiles_v2.h5')
sv_file = pd.read_hdf('../../SV_sample_data/data/analysis_public_os_data_merged_consensus_svs_v2.h5')
included_samples = pd.read_csv("../../SV_sample_data/data/final_samples.csv")

# Find and filter to samples that exist in all three datasets
valid_samples = (
    set(included_samples['tumor_normal_pair']) &
    set(sv_file['name']) &
    set(cnv_file['sample'])
)
sv_file = sv_file[sv_file['name'].isin(valid_samples)]
cnv_file = cnv_file[cnv_file['sample'].isin(valid_samples)]

print(f"Final sample count: {len(valid_samples)} samples")
print(f"SV: {sv_file['name'].nunique()}, CNV: {cnv_file['sample'].nunique()}")

# print samples in included_samples that are not in valid_samples
print("Samples without SV and/or CNV data:")
print(set(included_samples['tumor_normal_pair']) - valid_samples)

Final sample count: 228 samples
SV: 228, CNV: 228
Samples without SV and/or CNV data:
{'SJOS030591_D2-SJOS030591_G1', 'SJOS016016_D1-SJOS016016_G1', 'SJOS063833_D1-SJOS063833_G1', 'TARGET-40-PALECC-01A-TARGET-40-PALECC-10A', 'CCG1106_083_T1-CCG1106_083_WB', 'SJOS033930_D1-SJOS033930_G1', '17_439_00097_T1-17_439_00097_WB', 'SJOS030589_D4-SJOS030589_G2', 'SJOS063833_D2-SJOS063833_G1'}


In [21]:
def format_sv_data_for_shatterseek(sv_df):
    """
    Convert SV data to ShatterSeek format
    """
    sv_formatted = sv_df.copy()
    
    column_mapping = {
        'chr1': 'chrom1',
        'pos1': 'start1', 
        'chr2': 'chrom2',
        'pos2': 'end2',
        'class': 'original_class',
        'name': 'sample_id'
    }
    
    sv_formatted = sv_formatted.rename(columns=column_mapping)
    
    # Convert strand encoding (0 -> "+", 1 -> "-")
    sv_formatted['strand1'] = sv_formatted['str1'].map({0: '+', 1: '-'})
    sv_formatted['strand2'] = sv_formatted['str2'].map({0: '+', 1: '-'})
    
    # Remove "chr" prefix from chromosomes
    sv_formatted['chrom1'] = sv_formatted['chrom1'].astype(str).str.replace('chr', '', regex=False)
    sv_formatted['chrom2'] = sv_formatted['chrom2'].astype(str).str.replace('chr', '', regex=False)
    
    # chromosome name mapping for X & Y
    chrom_map = {
        '23': 'X',
        '24': 'Y'
    }
    sv_formatted['chrom1'] = sv_formatted['chrom1'].astype(str).map(lambda x: chrom_map.get(x, x))
    sv_formatted['chrom2'] = sv_formatted['chrom2'].astype(str).map(lambda x: chrom_map.get(x, x))
    
    # Remove Y chromosome variants
    before_y_filter = len(sv_formatted)
    sv_formatted = sv_formatted[~((sv_formatted['chrom1'] == 'Y') | (sv_formatted['chrom2'] == 'Y'))]
    after_y_filter = len(sv_formatted)
    if before_y_filter != after_y_filter:
        print(f"Removed {before_y_filter - after_y_filter} SVs involving Y chromosome")
    
    # Encode SV type based on strand orientations (as per ShatterSeek documentation)
    def encode_sv_type(row):
        strand1 = row['strand1']
        strand2 = row['strand2'] 
        chrom1 = row['chrom1']
        chrom2 = row['chrom2']
        original_class = row['original_class']
        
        # For translocations (different chromosomes), always use TRA
        if chrom1 != chrom2:
            return 'TRA'
        
        # For same chromosome, encode based on strand orientations
        if strand1 == '+' and strand2 == '-':
            return 'DEL'  # deletion-like (+/-)
        elif strand1 == '-' and strand2 == '+':
            return 'DUP'  # duplication-like (-/+)
        elif strand1 == '+' and strand2 == '+':
            return 'h2hINV'  # head-to-head inversion (+/+)
        elif strand1 == '-' and strand2 == '-':
            return 't2tINV'  # tail-to-tail inversion (-/-)
        else:
            print(f"Warning: Unusual strand pattern {strand1}/{strand2} for {original_class}")
            return original_class.upper()
    
    # Apply the strand-based encoding
    sv_formatted['svclass'] = sv_formatted.apply(encode_sv_type, axis=1)
    
    print("SV type encoding based on strands:")
    encoding_summary = sv_formatted.groupby(['strand1', 'strand2', 'svclass']).size().reset_index(name='count')
    print(encoding_summary)
    
    # Compare with original classifications
    comparison = sv_formatted.groupby(['original_class', 'svclass']).size().reset_index(name='count')
    print("\nOriginal vs. strand-based classification:")
    print(comparison)
    
    # Select only columns needed for ShatterSeek
    shatterseek_columns = [
        'sample_id', 'chrom1', 'start1', 'chrom2', 'end2', 'svclass',
        'strand1', 'strand2'
    ]
    
    sv_formatted = sv_formatted[shatterseek_columns]
    
    # Remove any rows with missing critical data
    critical_columns = ['chrom1', 'start1', 'chrom2', 'end2', 'svclass', 'strand1', 'strand2']
    before_filter = len(sv_formatted)
    sv_formatted = sv_formatted.dropna(subset=critical_columns)
    after_filter = len(sv_formatted)
    
    if before_filter != after_filter:
        print(f"Removed {before_filter - after_filter} SVs with missing data")
    
    return sv_formatted

sv_shatterseek = format_sv_data_for_shatterseek(sv_file)
sv_shatterseek.to_csv('../data/shatterseek_sv_data.csv', index=False)
sv_shatterseek.head()

Removed 131 SVs involving Y chromosome
SV type encoding based on strands:
  strand1 strand2 svclass  count
0       +       +     TRA   5075
1       +       +  h2hINV  10212
2       +       -     DEL  11960
3       +       -     TRA   3993
4       -       +     DUP  10615
5       -       +     TRA   5018
6       -       -     TRA   5728
7       -       -  t2tINV  10912

Original vs. strand-based classification:
  original_class svclass  count
0       deletion     DEL   6046
1      inter_chr     TRA  19814
2      inversion  h2hINV   4920
3      inversion  t2tINV   5042
4     long_range     DEL   5914
5     long_range     DUP   5547
6     long_range  h2hINV   5292
7     long_range  t2tINV   5870
8     tandem_dup     DUP   5068


,sample_id,chrom1,start1,chrom2,end2,svclass,strand1,strand2
0,09T02-09N01,1,3680543,1,3731731,DEL,+,-
1,09T02-09N01,1,25148899,1,25252735,DEL,+,-
2,09T02-09N01,1,124154195,1,124163217,DEL,+,-
3,09T02-09N01,1,160827717,1,160890810,DUP,-,+
4,09T02-09N01,1,246260290,19,6750721,TRA,-,-


In [ ]:
def merge_adjacent_same_total_cn(df):
    df = df.sort_values(["sample_id", "chromosome", "start", "end"]).copy()

    g = df.groupby(["sample_id", "chromosome"], sort=False)
    prev_end = g["end"].shift(1)
    prev_cn  = g["total_cn"].shift(1)

    touching_prev = (df["start"] == (prev_end + 1)) # adjacency
    same_cn_prev  = (df["total_cn"] == prev_cn)

    new_block = ~(touching_prev & same_cn_prev)
    df["_block"] = g["start"].transform(lambda s: new_block.loc[s.index].cumsum())

    merged = (
        df.groupby(["sample_id", "chromosome", "_block"], as_index=False)
          .agg(start=("start", "min"),
               end=("end", "max"),
               total_cn=("total_cn", "first"))
          .drop(columns=["_block"])
    )
    return merged

def format_cnv_data_for_shatterseek(cnv_df):
    """
    Convert CNV data to ShatterSeek format
    """
    cnv_formatted = cnv_df.copy()
    
    # Rename columns
    column_mapping = {
        'sample': 'sample_id',
        'Chromosome': 'chromosome',
        'Start.bp': 'start',
        'End.bp': 'end'
    }
    cnv_formatted = cnv_formatted.rename(columns=column_mapping)

    # Calculate total copy number from allele-specific copy numbers
    cnv_formatted['total_cn'] = cnv_formatted['rescaled.cn.a1'] + cnv_formatted['rescaled.cn.a2']
    
    # Remove "chr" prefix from chromosomes (ShatterSeek expects Ensembl notation)
    cnv_formatted['chromosome'] = cnv_formatted['chromosome'].astype(str).str.replace('chr', '', regex=False)
    
    # Remove Y chromosome variants
    before_y_filter = len(cnv_formatted)
    cnv_formatted = cnv_formatted[cnv_formatted['chromosome'] != 'Y']
    after_y_filter = len(cnv_formatted)
    if before_y_filter != after_y_filter:
        print(f"Removed {before_y_filter - after_y_filter} CNV segments from Y chromosome")
    
    # chromosome name mapping for X & Y
    chrom_map = {
        '23': 'X',
        '24': 'Y'
    }
    cnv_formatted['chromosome'] = cnv_formatted['chromosome'].astype(str).map(lambda x: chrom_map.get(x, x))
    
    # Select only the columns needed for ShatterSeek
    shatterseek_columns = ['sample_id', 'chromosome', 'start', 'end', 'total_cn']
    cnv_formatted = cnv_formatted[shatterseek_columns]
    
    # Remove any rows with missing critical data
    before_filter = len(cnv_formatted)
    cnv_formatted = cnv_formatted.dropna(subset=['chromosome', 'start', 'end', 'total_cn'])
    after_filter = len(cnv_formatted)
    if before_filter != after_filter:
        print(f"Removed {before_filter - after_filter} CNV segments with missing data")
    
    # Ensure chromosome is string format
    cnv_formatted['chromosome'] = cnv_formatted['chromosome'].astype(str)
    
    # Sort by sample, chromosome, and start position
    cnv_formatted = cnv_formatted.sort_values(['sample_id', 'chromosome', 'start'])

    return cnv_formatted

# consider rounding total CN to nearest integer? (for better plotting)
cnv_formatted = format_cnv_data_for_shatterseek(cnv_file)
print("Before merge:", len(cnv_formatted))
cnv_formatted = merge_adjacent_same_total_cn(cnv_formatted)
print("After merge:", len(cnv_formatted))
cnv_formatted.to_csv('../data/shatterseek_cnv_data.csv', index=False)
cnv_formatted.head()

Before merge: 102888
After merge: 89987


,sample_id,chromosome,start,end,total_cn
0,09T02-09N01,1,832000.0,3947999,4.0
1,09T02-09N01,1,3954000.0,13583999,4.0
2,09T02-09N01,1,13586000.0,16473999,4.0
3,09T02-09N01,1,16476000.0,16685999,4.0
4,09T02-09N01,1,16874000.0,16923999,6.0
